In [ ]:
# Installation des dépendances si nécessaire
# !pip install python-dotenv groq

import streamlit as st
import os
import json
import xmlrpc.client
import ssl
from dotenv import load_dotenv
from groq import Groq

# Chargement des variables d'environnement
load_dotenv()

# Initialisation du client Groq
client = Groq(api_key=os.getenv("api_key"))

print("✅ Environnement configuré et Clé API chargée.")

✅ Environnement configuré et Clé API chargée.


In [2]:
def odoo_fetch_sales(client_name=None):
    """
    Fonction outil : Se connecte à Odoo pour chercher les ventes.
    Peut filtrer par nom de client si spécifié.
    """
    url = os.getenv("ODOO_URL")
    db = os.getenv("ODOO_DB")
    username = os.getenv("ODOO_USER")
    password = os.getenv("ODOO_PASS")
    
    context = ssl._create_unverified_context()
    
    try:
        common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common", context=context)
        uid = common.authenticate(db, username, password, {})
        models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", context=context)
        
        # Définition du filtre
        domain = [('state', 'in', ['sale', 'done'])] # On ne prend que les ventes confirmées
        if client_name:
            domain.append(('partner_id', 'ilike', client_name))
            
        fields = ['name', 'date_order', 'partner_id', 'amount_total', 'state']
        sales = models.execute_kw(db, uid, password, 'sale.order', 'search_read', [domain], {'fields': fields, 'limit': 10})
        
        # Nettoyage pour l'IA
        for s in sales:
            if isinstance(s['partner_id'], list):
                s['client'] = s['partner_id'][1]
                del s['partner_id']
                
        return json.dumps(sales)
    except Exception as e:
        return f"Erreur lors de la connexion Odoo : {str(e)}"

print("🛠️ Outil 'odoo_fetch_sales' prêt.")

🛠️ Outil 'odoo_fetch_sales' prêt.


In [7]:
def odoo_fetch_stock():
    """Récupère l'état des stocks réels (Inventaire)."""
    url, db, username, password = os.getenv("ODOO_URL"), os.getenv("ODOO_DB"), os.getenv("ODOO_USER"), os.getenv("ODOO_PASS")
    common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common", context=ssl._create_unverified_context())
    uid = common.authenticate(db, username, password, {})
    models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", context=ssl._create_unverified_context())

    # On cherche les produits stockables avec leurs quantités
    fields = ['display_name', 'qty_available', 'virtual_available', 'list_price']
    stock_data = models.execute_kw(db, uid, password, 'product.product', 'search_read', 
                                   [[('type', '=', 'product')]], {'fields': fields, 'limit': 20})
    return json.dumps(stock_data)

def odoo_fetch_accounting_risk(client_name=None):
    """Vérifie les factures impayées pour évaluer le risque financier."""
    url, db, username, password = os.getenv("ODOO_URL"), os.getenv("ODOO_DB"), os.getenv("ODOO_USER"), os.getenv("ODOO_PASS")
    common = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/common", context=ssl._create_unverified_context())
    uid = common.authenticate(db, username, password, {})
    models = xmlrpc.client.ServerProxy(f"{url}/xmlrpc/2/object", context=ssl._create_unverified_context())

    # On cherche les factures postées mais non payées
    domain = [('state', '=', 'posted'), ('payment_state', 'not in', ['paid', 'reversed'])]
    if client_name:
        domain.append(('partner_id', 'ilike', client_name))
        
    fields = ['name', 'partner_id', 'amount_residual', 'invoice_date_due']
    invoices = models.execute_kw(db, uid, password, 'account.move', 'search_read', [domain], {'fields': fields, 'limit': 10})
    return json.dumps(invoices)

In [10]:
# Définition technique de l'outil pour le modèle LLM
tools = [
    {
        "type": "function",
        "function": {
            "name": "odoo_fetch_sales",
            "description": "Récupère les bons de commande. Paramètre client_name optionnel.",
            "parameters": {"type": "object", "properties": {"client_name": {"type": "string"}}, "required": []}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "odoo_fetch_stock",
            "description": "Vérifie les quantités en stock (Inventaire) et les prix des produits.",
            "parameters": {"type": "object", "properties": {}}
        }
    },
    {
        "type": "function",
        "function": {
            "name": "odoo_fetch_accounting_risk",
            "description": "Liste les factures impayées. Indispensable pour vérifier si un client est un bon payeur.",
            "parameters": {"type": "object", "properties": {"client_name": {"type": "string"}}, "required": []}
        }
    }
]


def run_expert_agent(user_prompt):
    messages = [
        {"role": "system", "content": "Tu es un Expert CFO/Logistique. Tu croises les données de Ventes, de Stock et de Comptabilité pour donner des conseils stratégiques. Si on te demande une analyse, utilise TOUS les outils nécessaires."},
        {"role": "user", "content": user_prompt}
    ]
    
    # Premier appel
    response = client.chat.completions.create(model="openai/gpt-oss-120b", messages=messages, tools=tools)
    
    response_message = response.choices[0].message
    tool_calls = response_message.tool_calls

    if tool_calls:
        messages.append(response_message)
        
        for tool_call in tool_calls:
            function_name = tool_call.function.name
            args = json.loads(tool_call.function.arguments)
            
            # Mapping des fonctions
            available_functions = {
                "odoo_fetch_sales": odoo_fetch_sales,
                "odoo_fetch_stock": odoo_fetch_stock,
                "odoo_fetch_accounting_risk": odoo_fetch_accounting_risk,
            }
            
            print(f"📡 L'Agent utilise l'outil : {function_name}...")
            result = available_functions[function_name](**args)
            
            messages.append({"tool_call_id": tool_call.id, "role": "tool", "name": function_name, "content": result})
        
        # Réponse finale consolidée
        final_res = client.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
        return final_res.choices[0].message.content
    
    return response_message.content

In [12]:
# Exemple de test
question = "comment a été passé la derniere semaine"
print(f"USER: {question}\n")
reponse = run_agent(question)
print(f"\nAGENT:\n{reponse}")

USER: comment a été passé la derniere semaine

📡 Action de l'Agent : Appel de odoo_fetch_sales({'client_name': ''})...

AGENT:
**Bilan de la dernière semaine (du 24 / 12 / 2025 au 30 / 12 / 2025)**  

| Date          | Nombre de ventes | Montant total (EUR) |
|---------------|------------------|---------------------|
| 29 / 01 / 2026 | 6                | 6 617,5 € |
| 30 / 01 / 2026 | 4                | 8 325,5 € |
| **Total**     | **10**           | **14 943,0 €** |

### Points clés

1. **Volume** – 10 bons de commande ont été enregistrés en 7 jours, ce qui représente une activité soutenue pour le service commercial.  
2. **Valeur** – Le chiffre d’affaires total s’élève à **14 943 €**, avec un ticket moyen de **≈ 1 494 €** par vente.  
3. **Client principal** – Tous les bons de commande concernent le même client, **Gemini Furniture**, ce qui indique une forte concentration de la facturation sur ce compte.  
4. **État des ventes** – Tous les enregistrements sont en état **« sale »**, 